# MEDIUM (11 nodes, 7 routes): T three-run spectrum notebook

This entry point preserves the original T/Tb three-run clock (4,000 / 6,000 / 8,000 frames) and its full configured model and scenario roster. Topology is supplied by the layered primary catalog; allocator strategy is held fixed for this comparison.


In [ ]:
import json, os
from pathlib import Path
from daqr.campaigns.spectrum_three import (
    BASE_FRAMES, FRAME_STEP, PHYSICS_MODEL, build_block_payload,
    fixed_allocation, run_cell,
)
TOPOLOGY_SCALE_M = 2
REPLAY = 'T'
CAPACITY_SCALES = (1, 1.5, 2)
RUNS = 3
OUTPUT_ROOT = Path(os.environ['SPECTRUM_OUTPUT_ROOT']).expanduser().resolve()
selection = os.environ.get('SPECTRUM_CAPACITY_SCALE')
selected_scales = tuple(s for s in CAPACITY_SCALES if selection is None or str(s) == selection)
if not selected_scales:
    raise ValueError('Requested capacity scale is absent from the notebook contract')
print({'topology_scale_m': TOPOLOGY_SCALE_M, 'routes': 3*TOPOLOGY_SCALE_M+1,
       'replay': REPLAY, 'capacity_scales': selected_scales, 'blocks': RUNS})


## Catalog and capacity preflight

Each block must have the configured route count and aligned contexts/rewards. T capacity follows the block horizon; Tb capacity stays anchored to the base horizon.


In [ ]:
for capacity_scale in selected_scales:
    for block_id in range(RUNS):
        frames = BASE_FRAMES + FRAME_STEP * block_id
        payload = build_block_payload(
            scale_m=TOPOLOGY_SCALE_M, replay=REPLAY, capacity_scale=capacity_scale,
            physics_model=PHYSICS_MODEL, current_frames=frames, base_seed=12345,
            qubit_cap=fixed_allocation(TOPOLOGY_SCALE_M), block_id=block_id,
        )
        assert len(payload['external_contexts']) == 3*TOPOLOGY_SCALE_M+1
        assert len(payload['external_rewards']) == len(payload['external_contexts'])
print('Catalog preflight passed')


## Execute complete experiments

Every selected capacity scale gets its own immutable output root and includes the full model and scenario roster across three blocks. This notebook records routing and allocation evidence; service equity requires an eligible demand ledger.


In [ ]:
receipts = {}
for capacity_scale in selected_scales:
    scale_label = str(capacity_scale).replace('.', '_')
    cell_root = OUTPUT_ROOT / f'm{TOPOLOGY_SCALE_M}-{REPLAY}-s{scale_label}-3'
    receipts[str(capacity_scale)] = str(run_cell(
        cell_root, scale_m=TOPOLOGY_SCALE_M, replay=REPLAY,
        capacity_scale=capacity_scale,
    ))
print(json.dumps(receipts, indent=2))
